# 🏠 도쿄 23구 주택 가격 예측 AI - 예시 노트북

이 노트북은 AutoML과 딥러닝을 사용하여 도쿄 23구 주택 가격을 예측하는 방법을 보여줍니다.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 프로젝트 모듈
from data.generate_tokyo_housing_data import generate_housing_data, add_derived_features, split_data
from models.deep_learning_layers import BasicMLP, ResidualNetwork, TabNetModel

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. 데이터 생성 및 탐색

In [ ]:
# 샘플 데이터 생성
df = generate_housing_data(n_samples=10000, random_seed=42)
df = add_derived_features(df)

print(f"데이터 크기: {df.shape}")
df.head()

In [ ]:
# 기본 통계
df.describe()

In [ ]:
# 구별 평균 가격 시각화
fig, ax = plt.subplots(figsize=(14, 8))
ward_prices = df.groupby('ward_jp')['price_man_yen'].mean().sort_values(ascending=True)
ward_prices.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('평균 가격 (만엔)')
ax.set_title('도쿄 23구별 평균 주택 가격')
plt.tight_layout()
plt.show()

In [ ]:
# 면적과 가격의 관계
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['floor_area_sqm'], df['price_man_yen'], alpha=0.3, s=5)
axes[0].set_xlabel('면적 (㎡)')
axes[0].set_ylabel('가격 (만엔)')
axes[0].set_title('면적 vs 가격')

axes[1].scatter(df['distance_to_station_min'], df['price_man_yen'], alpha=0.3, s=5)
axes[1].set_xlabel('역까지 거리 (분)')
axes[1].set_ylabel('가격 (만엔)')
axes[1].set_title('역까지 거리 vs 가격')

plt.tight_layout()
plt.show()

## 2. 딥러닝 레이어 이해하기

In [ ]:
import torch
import torch.nn as nn

# 기본 MLP 레이어 구조
print("=" * 60)
print("BasicMLP 아키텍처")
print("=" * 60)

mlp = BasicMLP(input_dim=15, hidden_dims=[256, 128, 64])
print(mlp)

In [ ]:
# ResidualNetwork 아키텍처
print("=" * 60)
print("ResidualNetwork 아키텍처")
print("=" * 60)

resnet = ResidualNetwork(input_dim=15, hidden_dim=128, num_blocks=4)
print(resnet)

In [ ]:
# TabNet 아키텍처
print("=" * 60)
print("TabNet 아키텍처")
print("=" * 60)

tabnet = TabNetModel(input_dim=15, n_d=64, n_a=64, n_steps=3)
print(tabnet)

## 3. 모델 테스트

In [ ]:
# 랜덤 입력으로 각 모델 테스트
batch_size = 32
input_dim = 15
x = torch.randn(batch_size, input_dim)

print(f"입력 형태: {x.shape}")
print()

# MLP
mlp_out = mlp(x)
print(f"MLP 출력: {mlp_out.shape}")

# ResNet
resnet_out = resnet(x)
print(f"ResNet 출력: {resnet_out.shape}")

# TabNet
tabnet_out, attention = tabnet(x)
print(f"TabNet 출력: {tabnet_out.shape}, 어텐션: {attention.shape}")

## 4. TabNet 어텐션 시각화

In [ ]:
# TabNet의 피처 어텐션 시각화
feature_names = ['floor_area', 'num_rooms', 'building_age', 'floor_num', 
                 'total_floors', 'distance_station', 'parking', 'balcony',
                 'security', 'corner', 'year', 'month', 'price_sqm',
                 'area_room', 'rel_floor']

# 어텐션 가중치 평균
avg_attention = attention.mean(dim=0).detach().numpy()

plt.figure(figsize=(12, 6))
plt.bar(range(len(avg_attention)), avg_attention, color='steelblue')
plt.xticks(range(len(feature_names)), feature_names, rotation=45, ha='right')
plt.xlabel('피처')
plt.ylabel('어텐션 가중치')
plt.title('TabNet 피처 어텐션 가중치')
plt.tight_layout()
plt.show()

## 5. 손실 함수 비교

In [ ]:
from models.deep_learning_layers import HuberLoss, MAPELoss, CombinedLoss

# 손실 함수들 비교
y_true = torch.tensor([1000., 2000., 3000., 4000., 5000.])
y_pred = torch.tensor([1100., 1800., 3200., 3800., 5500.])

mse = nn.MSELoss()
mae = nn.L1Loss()
huber = HuberLoss(delta=100)
mape = MAPELoss()

print("손실 함수 비교:")
print(f"  MSE:   {mse(y_pred, y_true).item():.2f}")
print(f"  MAE:   {mae(y_pred, y_true).item():.2f}")
print(f"  Huber: {huber(y_pred, y_true).item():.2f}")
print(f"  MAPE:  {mape(y_pred, y_true).item():.2f}%")

## 6. 다음 단계

전체 학습을 실행하려면 터미널에서:

```bash
# AutoML 학습
python train_automl.py --method ensemble --n_trials 100

# 딥러닝 학습
python train_deep_learning.py --model transformer --epochs 100
```